# 05 — XAI: Atenção Agregada e Teste de Fidelidade por Deleção

**Objetivo:** Aplicar técnicas de Explainable AI (XAI) ao EmoTransformer treinado:

1. **Atenção Agregada (Attention Rollout):** Agregar os pesos de atenção de todas as
   camadas para gerar um mapa de importância por frame temporal.
2. **Importância por Landmark/Feature:** Usando gradientes × atenção para identificar
   quais features (landmarks) são mais relevantes.
3. **Teste de Fidelidade por Deleção:** Mascarar os top-k frames mais importantes
   e medir a queda na probabilidade/logit do modelo.

**Outputs:**
- `reports/figures/xai_temporal_curve.png`
- `reports/figures/xai_landmark_importance.png`
- `runs/poc_v1/metrics/xai_fidelity_deletion.csv`

In [ ]:
import sys, os, json

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from src.models import EmoTransformer
from src.metrics_utils import fix_seed
from src.ravdess_utils import EMOTION_MAP, EMOTION_LABELS

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
SEED = 42
fix_seed(SEED)

print(f"Raiz: {ROOT}")

## 5.1 Carregar Modelo e Dados

In [ ]:
# Paths
DATA_DIR = os.path.join(ROOT, 'data', 'ravdess_landmarks_kaggle', '01_processed_T100')
SPLIT_DIR = os.path.join(ROOT, 'data', 'ravdess_landmarks_kaggle', '02_splits')
CKPT_DIR = os.path.join(ROOT, 'runs', 'poc_v1', 'checkpoints')
METRICS_DIR = os.path.join(ROOT, 'runs', 'poc_v1', 'metrics')
FIGS_DIR = os.path.join(ROOT, 'reports', 'figures')

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

In [ ]:
# Carregar checkpoint
ckpt = torch.load(os.path.join(CKPT_DIR, 'best_transformer.pt'), map_location='cpu', weights_only=False)
config = ckpt['config']
mu = np.array(ckpt['normalization']['mu'])
sigma = np.array(ckpt['normalization']['sigma'])

print(f"Config do modelo: {config}")

# Reconstruir modelo
model = EmoTransformer(**config)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Modelo carregado (epoch {ckpt['best_epoch']})")

In [ ]:
# Carregar dados
data = np.load(os.path.join(DATA_DIR, 'dataset_T100.npz'), allow_pickle=True)
X_all = data['X']
y_all = data['y']

# Carregar split
with open(os.path.join(SPLIT_DIR, 'split_actor_holdout.json'), 'r') as f:
    split = json.load(f)

test_idx = split['test_indices']
X_test = X_all[test_idx]
y_test = y_all[test_idx]

# Normalizar
X_test_norm = (X_test - mu) / sigma

D = X_test_norm.shape[2]
T = X_test_norm.shape[1]

# Determinar class names
unique_classes = sorted(np.unique(y_all))
class_names = [EMOTION_MAP.get(c + 1, f"emo_{c}") for c in unique_classes]

print(f"Test set: {X_test_norm.shape}")
print(f"Classes: {class_names}")

## 5.2 Atenção Agregada (Attention Rollout)

O Attention Rollout acumula as matrizes de atenção de cada camada,
propagando atenção residual. Extraímos a atenção do CLS token
para cada frame, gerando importância temporal.

In [ ]:
def attention_rollout(attn_weights_list):
    """
    Calcula Attention Rollout a partir de uma lista de attention weights.
    
    Args:
        attn_weights_list: lista de tensors (B, n_heads, seq_len, seq_len)
    
    Returns:
        rollout: tensor (B, seq_len, seq_len) — atenção acumulada
    """
    # Média sobre heads
    result = None
    for attn in attn_weights_list:
        # attn: (B, n_heads, S, S) -> média sobre heads -> (B, S, S)
        attn_mean = attn.mean(dim=1)
        
        # Adicionar conexão residual (identidade)
        I = torch.eye(attn_mean.size(-1)).unsqueeze(0).expand_as(attn_mean)
        attn_res = 0.5 * attn_mean + 0.5 * I
        
        # Normalizar linhas
        attn_res = attn_res / attn_res.sum(dim=-1, keepdim=True)
        
        if result is None:
            result = attn_res
        else:
            result = torch.bmm(attn_res, result)
    
    return result


def get_temporal_importance(model, x_tensor):
    """
    Extrai importância temporal por frame usando Attention Rollout.
    
    Args:
        model: EmoTransformer
        x_tensor: (1, T, D)
    
    Returns:
        frame_importance: (T,) — importância normalizada por frame
        logits: (1, n_classes)
    """
    model.eval()
    with torch.no_grad():
        logits, attn_list = model(x_tensor, return_attention=True)
    
    # Attention rollout
    rollout = attention_rollout(attn_list)  # (1, S, S) onde S = T+1 (CLS + T frames)
    
    # Extrair atenção do CLS (posição 0) para cada frame (posições 1:T+1)
    cls_attn = rollout[0, 0, 1:]  # (T,)
    
    # Normalizar para [0, 1]
    cls_attn = cls_attn / cls_attn.max() if cls_attn.max() > 0 else cls_attn
    
    return cls_attn.numpy(), logits

print("Funções de XAI definidas.")

## 5.3 Extrair Importância Temporal para Exemplos

In [ ]:
# Selecionar exemplos: pelo menos 2 emoções diferentes
# Vamos pegar 1 amostra corretamente classificada de cada emoção disponível no test

# Primeiro, obter predições para todo o test set
examples = []

with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test_norm)
    all_logits = model(X_test_tensor)
    all_preds = all_logits.argmax(dim=1).numpy()

# Para cada emoção no test, pegar a primeira amostra correta
for emo_code in sorted(np.unique(y_test)):
    mask = (y_test == emo_code) & (all_preds == emo_code)
    if mask.any():
        idx = np.where(mask)[0][0]
        examples.append({
            'idx': idx,
            'emotion_code': emo_code,
            'emotion_label': EMOTION_MAP.get(emo_code + 1, f"emo_{emo_code}"),
        })

print(f"Exemplos selecionados ({len(examples)} emoções):")
for ex in examples:
    print(f"  idx={ex['idx']}, {ex['emotion_label']} (code={ex['emotion_code']})")

In [ ]:
# Extrair importância temporal para cada exemplo
importances = []

for ex in examples:
    x_sample = torch.FloatTensor(X_test_norm[ex['idx']:ex['idx']+1])  # (1, T, D)
    frame_imp, logits = get_temporal_importance(model, x_sample)
    
    prob = F.softmax(logits, dim=1)[0]
    pred_class = prob.argmax().item()
    pred_prob = prob[pred_class].item()
    
    importances.append({
        'frame_importance': frame_imp,
        'pred_class': pred_class,
        'pred_prob': pred_prob,
        **ex,
    })
    
    print(f"{ex['emotion_label']}: pred={EMOTION_MAP.get(pred_class+1, '?')} (p={pred_prob:.3f})")

## 5.4 Figura: Curva de Importância Temporal

In [ ]:
# Visualizar importância temporal para múltiplas emoções
n_examples = min(len(importances), 4)  # até 4 exemplos
fig, axes = plt.subplots(n_examples, 1, figsize=(14, 3 * n_examples), sharex=True)

if n_examples == 1:
    axes = [axes]

colors_xai = plt.cm.Set2(np.linspace(0, 1, 8))

for i, imp in enumerate(importances[:n_examples]):
    ax = axes[i]
    frames = np.arange(T)
    importance = imp['frame_importance']
    
    # Barplot com cores
    color = colors_xai[imp['emotion_code'] % len(colors_xai)]
    ax.bar(frames, importance, color=color, alpha=0.7, width=1.0)
    ax.set_ylabel('Importância')
    ax.set_title(f"{imp['emotion_label']} (pred prob={imp['pred_prob']:.3f})")
    
    # Marcar top-10 frames
    top_k = 10
    top_frames = np.argsort(importance)[-top_k:]
    ax.scatter(top_frames, importance[top_frames], color='red', zorder=5, s=20, label=f'Top-{top_k}')
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Frame (t=0..99)')
plt.suptitle('Importância Temporal por Frame (Attention Rollout)', fontsize=13)
plt.tight_layout()

fig_path = os.path.join(FIGS_DIR, 'xai_temporal_curve.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvo: {fig_path}")

## 5.5 Importância por Feature/Landmark

Usamos gradiente × input para estimar a importância de cada feature (landmark coordinate).
Agregar sobre frames e amostras para obter um ranking global.

In [ ]:
def compute_feature_importance_gradient(model, x_tensor, target_class):
    """
    Calcula importância de features via gradient * input.
    
    Args:
        model: EmoTransformer
        x_tensor: (1, T, D) — requer grad
        target_class: int
    
    Returns:
        feature_importance: (D,) — importância média absoluta por feature
    """
    model.eval()
    x_tensor = x_tensor.detach().clone().requires_grad_(True)
    
    logits = model(x_tensor)  # (1, n_classes)
    target_logit = logits[0, target_class]
    
    target_logit.backward()
    
    # gradient * input
    grad_input = (x_tensor.grad * x_tensor).abs()  # (1, T, D)
    
    # Média sobre T (importância por feature)
    feature_imp = grad_input[0].mean(dim=0).detach().numpy()  # (D,)
    
    return feature_imp

print("Função de importância por feature definida.")

In [ ]:
# Calcular importância de features para múltiplas amostras do test
n_samples_for_feat = min(50, len(X_test_norm))  # usar até 50 amostras

all_feat_imp = []
for i in range(n_samples_for_feat):
    x_t = torch.FloatTensor(X_test_norm[i:i+1])
    true_class = int(y_test[i])
    fimp = compute_feature_importance_gradient(model, x_t, true_class)
    all_feat_imp.append(fimp)

# Média global
mean_feat_imp = np.mean(all_feat_imp, axis=0)  # (D,)
print(f"Feature importance shape: {mean_feat_imp.shape}")
print(f"Top-5 features mais importantes: {np.argsort(mean_feat_imp)[-5:][::-1]}")

In [ ]:
# Visualizar importância por feature (top-20)
top_n = min(20, D)
sorted_idx = np.argsort(mean_feat_imp)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(top_n), mean_feat_imp[sorted_idx][::-1], color='teal', edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels([f'Feature {idx}' for idx in sorted_idx[::-1]])
ax.set_xlabel('Importância Média (|grad × input|)')
ax.set_title(f'Top-{top_n} Features Mais Importantes (Gradient × Input)')

plt.tight_layout()
fig_path = os.path.join(FIGS_DIR, 'xai_landmark_importance.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvo: {fig_path}")

## 5.6 Teste de Fidelidade por Deleção

Mascaramos progressivamente os top-k frames mais importantes (segundo o attention rollout)
e medimos a queda na probabilidade da classe predita. Se a explicação é fiel,
a probabilidade deve cair mais rápido quando deletamos os frames mais importantes
do que quando deletamos frames aleatórios.

In [ ]:
def deletion_test(model, x_tensor, frame_importance, k_values, mask_value=0.0):
    """
    Testa fidelidade por deleção: mascara top-k frames e mede queda na probabilidade.
    
    Args:
        model: EmoTransformer
        x_tensor: (1, T, D)
        frame_importance: (T,) — importância por frame
        k_values: lista de k (quantos frames mascarar)
        mask_value: valor para mascarar (0.0 = zerar)
    
    Returns:
        results: lista de dicts com k, prob_original, prob_masked, prob_drop
    """
    model.eval()
    results = []
    
    # Predição original
    with torch.no_grad():
        logits_orig = model(x_tensor)
        probs_orig = F.softmax(logits_orig, dim=1)[0]
        pred_class = probs_orig.argmax().item()
        prob_orig = probs_orig[pred_class].item()
    
    # Ordenar frames por importância (mais importante primeiro)
    sorted_frames = np.argsort(frame_importance)[::-1]
    
    for k in k_values:
        # Mascarar top-k frames (importância)
        x_masked = x_tensor.clone()
        frames_to_mask = sorted_frames[:k]
        x_masked[0, frames_to_mask, :] = mask_value
        
        with torch.no_grad():
            logits_masked = model(x_masked)
            probs_masked = F.softmax(logits_masked, dim=1)[0]
            prob_masked = probs_masked[pred_class].item()
        
        # Random baseline: mascarar k frames aleatórios
        rng = np.random.RandomState(42)
        random_frames = rng.choice(T, size=k, replace=False)
        x_random = x_tensor.clone()
        x_random[0, random_frames, :] = mask_value
        
        with torch.no_grad():
            logits_random = model(x_random)
            probs_random = F.softmax(logits_random, dim=1)[0]
            prob_random = probs_random[pred_class].item()
        
        results.append({
            'k': k,
            'k_pct': round(100 * k / T, 1),
            'prob_original': round(prob_orig, 4),
            'prob_after_top_k_deletion': round(prob_masked, 4),
            'prob_after_random_deletion': round(prob_random, 4),
            'drop_top_k': round(prob_orig - prob_masked, 4),
            'drop_random': round(prob_orig - prob_random, 4),
        })
    
    return results, pred_class

print("Função de deletion test definida.")

In [ ]:
# Rodar deletion test para exemplos selecionados
k_values = [5, 10, 15, 20, 30, 40, 50]

all_deletion_results = []

for imp in importances[:4]:  # até 4 exemplos
    idx = imp['idx']
    x_t = torch.FloatTensor(X_test_norm[idx:idx+1])
    frame_imp = imp['frame_importance']
    
    results, pred_class = deletion_test(model, x_t, frame_imp, k_values)
    
    for r in results:
        r['sample_idx'] = idx
        r['true_emotion'] = imp['emotion_label']
        r['pred_emotion'] = EMOTION_MAP.get(pred_class + 1, f"emo_{pred_class}")
    
    all_deletion_results.extend(results)
    
    print(f"\n{imp['emotion_label']} (idx={idx}):")
    for r in results:
        print(f"  k={r['k']:3d} ({r['k_pct']:5.1f}%): "
              f"prob_orig={r['prob_original']:.3f} -> "
              f"top-k={r['prob_after_top_k_deletion']:.3f} (drop={r['drop_top_k']:.3f}) | "
              f"random={r['prob_after_random_deletion']:.3f} (drop={r['drop_random']:.3f})")

In [ ]:
# Salvar métricas XAI
deletion_df = pd.DataFrame(all_deletion_results)
xai_path = os.path.join(METRICS_DIR, 'xai_fidelity_deletion.csv')
deletion_df.to_csv(xai_path, index=False)
print(f"Métricas XAI salvas: {xai_path}")

deletion_df.head(15)

## 5.7 Visualização: Curva de Deleção

In [ ]:
# Agregar resultados (média sobre amostras)
agg = deletion_df.groupby('k').agg({
    'prob_original': 'mean',
    'prob_after_top_k_deletion': 'mean',
    'prob_after_random_deletion': 'mean',
    'drop_top_k': 'mean',
    'drop_random': 'mean',
    'k_pct': 'first',
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva de probabilidade
axes[0].plot(agg['k'], agg['prob_after_top_k_deletion'], 'ro-', label='Top-k deletion', linewidth=2)
axes[0].plot(agg['k'], agg['prob_after_random_deletion'], 'b^--', label='Random deletion', linewidth=2)
axes[0].axhline(y=agg['prob_original'].iloc[0], color='gray', linestyle=':', label='Original')
axes[0].set_xlabel('k (frames deletados)')
axes[0].set_ylabel('Probabilidade da classe predita')
axes[0].set_title('Teste de Fidelidade por Deleção')
axes[0].legend()

# Curva de queda
axes[1].plot(agg['k'], agg['drop_top_k'], 'ro-', label='Top-k deletion', linewidth=2)
axes[1].plot(agg['k'], agg['drop_random'], 'b^--', label='Random deletion', linewidth=2)
axes[1].set_xlabel('k (frames deletados)')
axes[1].set_ylabel('Queda na probabilidade')
axes[1].set_title('Queda de Probabilidade: Top-k vs Random')
axes[1].legend()

plt.suptitle('Teste de Fidelidade (Deletion Test) — Attention Rollout', fontsize=13)
plt.tight_layout()

fig_path = os.path.join(FIGS_DIR, 'xai_deletion_curve.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvo: {fig_path}")

## 5.8 Resumo XAI

In [ ]:
print("\n" + "=" * 70)
print("RESUMO XAI")
print("=" * 70)
print(f"Método: Attention Rollout (atenção agregada multi-camada)")
print(f"Importância por Feature: Gradient × Input")
print(f"Teste de Fidelidade: Deletion Test (top-k vs random)")
print(f"")
print(f"Resultados da deleção (médias):")
for _, row in agg.iterrows():
    print(f"  k={int(row['k']):3d} ({row['k_pct']:.0f}%): "
          f"drop_top_k={row['drop_top_k']:.3f}, "
          f"drop_random={row['drop_random']:.3f}, "
          f"ratio={row['drop_top_k']/max(row['drop_random'], 1e-6):.2f}x")
print(f"")
print(f"Figuras geradas:")
print(f"  - reports/figures/xai_temporal_curve.png")
print(f"  - reports/figures/xai_landmark_importance.png")
print(f"  - reports/figures/xai_deletion_curve.png")
print(f"  - runs/poc_v1/metrics/xai_fidelity_deletion.csv")
print("=" * 70)
print("\nNotebook 05 concluído com sucesso!")